## inference for proportions 1

In [ ]:
# 비율 추론. phat, se, 신뢰구간, 검정통계량, pvalue.

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import scipy as sci
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest


### 데이터 (엑셀, csv 파일, 인터넷, 원격 파일) 읽기

In [ ]:
# 1. 데이터 준비. 읽기. 생성.
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat = pd.read_csv(dat_url)
df_dat #.head()

,i,gender,ht
0,1,1,159.9
1,2,2,157.5
2,3,2,158.0
3,4,2,154.2
4,5,1,163.3
...,...,...,...
20123,20124,2,157.3
20124,20125,1,175.9
20125,20126,1,175.6
20126,20127,2,158.0


### 변수 생성, 리네임, 일부 추출

In [ ]:
df_dat['gn'] = df_dat['gender'].to_numpy()
gn = df_dat['gn']
df_dat['hgt'] = df_dat['ht'].to_numpy()
hgt = df_dat['hgt']
n_ttl = len(df_dat)

n_frac = 0.025                   # (2.5%, 대략 500개)
n_smpld = int( n_frac * n_ttl)   # 추출 갯수, 비율,

# 단순 무작위 추출 (2.5%, 대략 500개)
df_smpl = df_dat.sample(n_smpld, replace=False, random_state=42)
df_smpl

,i,gender,ht,gn,hgt
15561,15562,1,170.9,1,170.9
13056,13057,2,144.4,2,144.4
5702,5703,1,170.5,1,170.5
3062,3063,2,158.5,2,158.5
9199,9200,2,155.1,2,155.1
...,...,...,...,...,...
19666,19667,2,153.4,2,153.4
4442,4443,1,168.5,1,168.5
14814,14815,1,174.0,1,174.0
4339,4340,1,162.0,1,162.0


### 자료셋, 그룹 구분
남 비율, 남여 1:1 검사 예정, 즉 남 비율=0.5 검정

###  남 비율(표본비율) $ \hat{ p } $
 $ \hat p ={\sum X_i \over n} $, $\ X_i $= 0 또는 1 .  

In [ ]:
# 성별 변수 코딩을 1, 0으로 변환
gndr = (gn == 1).astype(int)    # gn = 1, 2; gndr = 1, 0. 1/0으로 전환.

p_hat = gndr.mean()   # 1의 비율임. 즉, 남 비율.
print("표본비율 추정치: " , p_hat)


표본비율 추정치:  0.44415739268680443


### 남 비율(표본비율)의 분산, 표준오차
$ V = { p(1-p) \over n }, \ $
$ V^{1/2} = \sqrt{ p(1-p) \over n } $ a.k.a SE

In [ ]:
var_bernoulli = p_hat * (1 - p_hat)
print(" variance of Bernoulli : " , var_bernoulli )

var_phat =  var_bernoulli / n_smpld
print(" Variance of phat : " , var_phat )

se_phat = var_phat**(1/2)
print(" Standard error of phat : " , se_phat )

se_max = (1/2 * 1/2 / n_smpld )**(1/2)
print(" max SE of phat : " , se_max )

print(" 옵션 1, 2를 이용한 계산. 아래 참조.")

 variance of Bernoulli :  0.24688160320846422
 Variance of phat :  0.0004908182966371058
 Standard error of phat :  0.02215441934777587
 max SE of phat :  0.02229389810338549
 옵션 1, 2를 이용한 계산. 아래 참조.


### 남 비율 추론 1. 신뢰구간

$ 100(1-\alpha ) $% 신뢰구간
$$ \hat p \pm z_{\alpha\over 2} \times SE  $$
$ SE = \sqrt{p (1-p) \over n } $

$ SE $ 추정 옵션. 1) $ \hat p $ 사용. 2) $ 1\over 2 $ 사용, 3) $ p_0 $ 사용

#### 유의수준, 신뢰수준, 임계치

In [ ]:
# 유의(신뢰)수준, 정규분포 임계치, 양방향, 우측값 (양수)
alpha = 0.05                          # significance level, two side.
confidence_level = 100*(1 - alpha)           # confidence level, two side.

zcv_l = stats.norm.ppf( alpha/2 )  # critical value on the left, two side
zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side

me_max = se_max * zcv_r
ci_left = p_hat - zcv_r * se_max
ci_right = p_hat + zcv_r * se_max

print("유의수준 : ", alpha )
print("신뢰수준 : ", confidence_level )
print("2 임계치 : ", zcv_l, zcv_r )
print(" SE_max : ", se_max )
print("최대허용오차 ", me_max )
print(" phat : ", p_hat )
print("신뢰구간(최대) : ", ci_left, ci_right )

유의수준 :  0.05
신뢰수준 :  95.0
2 임계치 :  -1.9599639845400545 1.959963984540054
 SE_max :  0.02229389810338549
최대허용오차  0.04369523735764138
 phat :  0.44415739268680443
신뢰구간(최대) :  0.40046215532916307 0.4878526300444458


### 남 비율 추론 2. 검정통계량

##### 가설검정, 검정통계량
가설(양측)
\begin{align}
 H_0 &: p = p_0 \quad \text{vs.} \quad
 H_A : p \ne p_0
\end{align}
검정통계량
\begin{align}
 T_0 & = { \hat p - p_0 \over SE_0}  \\
 & \approx N(0,1)
\end{align}

##### $ H_0 : p=p_0 \ vs. \ H_A: p \ne p_0 $

In [ ]:
# 귀무가설 H0: p=p0, 대립가설 HA: p is not p0

p_zero = 0.5
print(f" H0: p = {p_zero} vs. HA: p is not { p_zero} " )

 H0: p = 0.5 vs. HA: p is not 0.5 


##### $ T_0 $
\begin{align}
 T_0 & = { \hat p - p_0 \over SE_0}   
\end{align}

In [ ]:
# SE 옵션 3  .
se_zero = np.sqrt( p_zero * (1 - p_zero) / n_smpld )  # 옵션 3.
t_0 = np.abs( ( p_hat - p_zero ) / se_zero )       # 옵션 3.

#                   alpha = 0.05                          # significance level, two side.
#confidence_level = 100*(1 - alpha)           # confidence level, two side.
#zcv_l = stats.norm.ppf( alpha/2 )  # critical value on the left, two side
#zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side

print( f"reject H0 if test statistic, t_0 = {t_0} > critical value zcv_r = {zcv_r} at significance level {alpha}" )
result_test = " 'Reject H0' " if t_0 > zcv_r else " 'Fail to reject H0' "   # 이건 되는 군.
print(f" 검정통계치,    임계치(유의수준={alpha}),    검정결과 ")
print( t_0 , zcv_r, result_test)

reject H0 if test statistic, t_0 = 2.504838187302716 > critical value zcv_r = 1.959963984540054 at significance level 0.05
 검정통계치,    임계치(유의수준=0.05),    검정결과 
2.504838187302716 1.959963984540054  'Reject H0' 


### 남 비율 추론 3. $ p $ value

#### $  p $-value
$ 2\times P(T_0 > |t_0| ) $

In [ ]:
# 이제 p값

p_val = 2*( 1 - stats.norm.cdf( t_0 ) )

print(f" p값 = { p_val } " )
f"reject H0 if p값 < {alpha}"

 p값 = 0.012250742540926174 


'reject H0 if p값 < 0.05'